In [1]:
!pip install kfp-kubernetes==1.4.0 kserve==0.15.2 kubernetes==26.1.0

  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tritonclient 2.59.0 requires urllib3>=2.0.7, but you have urllib3 1.26.20 which is incompatible.
tensorflow 2.15.1 requires numpy<2.0.0,>=1.23.5, but you have numpy 2.2.6 which is incompatible.
feast 0.40.1 requires numpy<2,>=1.22, but you have numpy 2.2.6 which is incompatible.


In [15]:
import kfp
from kfp import dsl
from kfp import kubernetes
from kfp import local
from kfp.dsl import Input, Output, Dataset, Model, Artifact

# TIP: you may need to authenticate with the KFP instance
# local.init(runner=local.SubprocessRunner())
kfp_client = kfp.Client()

In [3]:
current_sc = os.popen("kubectl get pvc user-pvc -o=jsonpath='{.spec.storageClassName}'").read()
namespace_cur = os.popen("kubectl get pvc user-pvc -o=jsonpath='{.metadata.namespace}'").read()
print(namespace_cur)
print(current_sc)

geun-tak-roh-2e590eb8
gl4f-filesystem


In [11]:
@dsl.component(
    # base_image="geuntakroh/kfp-test:v0.7",
)
def check_location():
    import os
    import subprocess
    subprocess.run(['pwd'])
    subprocess.run(['whoami'])
    subprocess.run(['ls','-al'])
    subprocess.run(['df','-Th'])
    subprocess.run(['ls','-al','/data/'])
    subprocess.run(['ls','-al','/'])

@dsl.pipeline()
def check_location_pipe():
    task1 = check_location()
    kfp.kubernetes.add_pod_annotation(
        task=task1,
        annotation_key="hpe-ezua/add-auth-token",
        annotation_value="true"
    )

    kfp.kubernetes.set_image_pull_policy(
        task=task1,
        # policy="Always"
        policy="IfNotPresent"
    )
    kubernetes.mount_pvc(
        task1,
        pvc_name='user-pvc',
        mount_path='/data',
    )

kfp_client.create_run_from_pipeline_func(
    check_location_pipe,
    arguments={},
    experiment_name="test-rhgt-exp",
    enable_caching=False # failed: failed to create PVC and publish execution createpvc: failed to create cache entrty for create pvc: failed to create task: rpc error: code = InvalidArgument desc = Failed to create a new task due to validation error: Invalid input error: Invalid task: must specify FingerPrint
    # enable_caching=True
)

/opt/conda/lib/python3.11/site-packages/kfp/dsl/component_decorator.py:119: FutureWarning: Python 3.7 has reached end-of-life. The default base_image used by the @dsl.component decorator will switch from 'python:3.7' to 'python:3.8' on April 23, 2024. To ensure your existing components work with versions of the KFP SDK released after that date, you should provide an explicit base_image argument and ensure your component works as intended on Python 3.8.
  return component_factory.create_component_from_func(


RunPipelineResult(run_id=0c1efa3e-db59-405e-8ae9-6a23a3a8a3e0)

In [122]:
@dsl.component(
    base_image="geuntakroh/kfp-test:v0.9",
)
def inference_video(model_url:str,vehicle_conf: float,license_conf: float,inference_results_csv: Output[Artifact]):
    from ultralytics import YOLO
    import easyocr
    import cv2 as cv
    import matplotlib.pyplot as plt
    import pandas as pd
    from urllib.parse import urlparse
    import string

    parts = urlparse(model_url)
    split = parts.netloc.split('.')[0]
    svc_name, namespace = split.split('-predictor-')
    url_for_yolo = f"http://{svc_name}-predictor-00001.{namespace}.svc.cluster.local"
    # print(url_for_yolo)
    vehicle_tracker = YOLO(url_for_yolo + "/vehicle_detector", task='detect')
    license_detector = YOLO(url_for_yolo + "/license_detector", task='detect')
    plate_reader = easyocr.Reader(['en'],gpu=False)
    
    results = []
    video_path = '/data/sample_videos/sample.mp4'
    # vehicle_conf = 0.5
    # license_conf = 0.5
    vehicles_id = [2,3,5,7]

    # Process video
    cap = cv.VideoCapture(video_path)
    fps = cap.get(cv.CAP_PROP_FPS)
    total_frame = cap.get(cv.CAP_PROP_FRAME_COUNT)
    # thres_frame = total_frame*ratio
    frame_number = -1

    dict_char_to_int = {'O': '0',
                        'I': '1',
                        'J': '3',
                        'A': '4',
                        'G': '6',
                        'S': '5'}
    
    dict_int_to_char = {'0': 'O',
                        '1': 'I',
                        '3': 'J',
                        '4': 'A',
                        '6': 'G',
                        '5': 'S'}
    
    def license_complies_format(text):
        # True if the license plate complies with the format, False otherwise.
        if len(text) != 7:
            return False
    
        if (text[0] in string.ascii_uppercase or text[0] in dict_int_to_char.keys()) and \
           (text[1] in string.ascii_uppercase or text[1] in dict_int_to_char.keys()) and \
           (text[2] in ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9'] or text[2] in dict_char_to_int.keys()) and \
           (text[3] in ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9'] or text[3] in dict_char_to_int.keys()) and \
           (text[4] in string.ascii_uppercase or text[4] in dict_int_to_char.keys()) and \
           (text[5] in string.ascii_uppercase or text[5] in dict_int_to_char.keys()) and \
           (text[6] in string.ascii_uppercase or text[6] in dict_int_to_char.keys()):
            return True
        else:
            return False
    
    def format_license(text):
        license_plate_ = ''
        mapping = {
            0: dict_int_to_char,
            1: dict_int_to_char,
            2: dict_char_to_int, 
            3: dict_char_to_int,
            4: dict_int_to_char, 
            5: dict_int_to_char, 
            6: dict_int_to_char
        }
        for j in [0, 1, 2, 3, 4, 5, 6]:
            if text[j] in mapping[j].keys():
                license_plate_ += mapping[j][text[j]]
            else:
                license_plate_ += text[j]
    
        return license_plate_

    def reformat_license_number(detections):
        for detection in detections:
            bbox, text, score = detection
            text = text.upper().replace(' ', '')
            if license_complies_format(text):
                return format_license(text), score
    
        return None, None

    
    try:
        # max_frame_num = 10
        while cap.isOpened():
            frame_number += 1
            ret, input_image = cap.read()
            # if not ret or frame_number > max_frame_num:
            if not ret:
                break
            print(f"Do Inference at frame_number: {frame_number}")
            frame_results = [] # change return value type as list
        
            vehicle_results = vehicle_tracker.track(input_image, persist=True,conf=vehicle_conf,classes=vehicles_id)[0]
            for vehicle_result in vehicle_results.boxes.data.tolist():
                x1, y1, x2, y2, track_id, score, class_id = vehicle_result
                
                vehicle_bounding_boxes = []
                vehicle_bounding_boxes.append([x1, y1, x2, y2, track_id, score])
                for bbox in vehicle_bounding_boxes: 
                    roi = input_image[int(y1):int(y2), int(x1):int(x2)] # crop the vehicle
                    license_plates = license_detector(roi,conf=license_conf)[0]
                    for license_plate in license_plates.boxes.data.tolist():
                        plate_x1, plate_y1, plate_x2, plate_y2, plate_score, _ = license_plate
                        # crop license plate
                        plate = roi[int(plate_y1):int(plate_y2), int(plate_x1):int(plate_x2)]
                        # de-colorize
                        plate_gray = cv.cvtColor(plate, cv.COLOR_BGR2GRAY)
                        # posterize
                        _, plate_treshold = cv.threshold(plate_gray, 64, 255, cv.THRESH_BINARY_INV)
        
                        ### try rgb and gray and get the best
                        rgb_detections = plate_reader.readtext(plate)
                        gray_detections = plate_reader.readtext(plate_treshold)
                        rgb_text, rgb_score = reformat_license_number(rgb_detections)
                        gray_text, gray_score = reformat_license_number(gray_detections)
                        
                        candidates = [(rgb_text,rgb_score),(gray_text,gray_score)]
                        valid_candidates = [(text,score) for text,score in candidates if score is not None]
                        lic_text, lic_score = max(valid_candidates,default=(None,None))
                       
                        frame_results.append({
                            "frame_number": frame_number,
                            "track_id": track_id,
                            "vehicle_bbox": [x1, y1, x2, y2],
                            "vehicle_bbox_score": score,
                            "lp_bbox": [plate_x1 , plate_y1, plate_x2, plate_y2],
                            "lp_bbox_score": plate_score,
                            "lp_number": lic_text,
                            "lp_text_score": lic_score
                        })
            results.extend(frame_results)

    
    finally:    
        # Clean up input video file
        cap.release()
        # Save results as csv
        pd.DataFrame(results).to_csv(inference_results_csv.path ,index=False)


In [123]:
@dsl.component(
    base_image="geuntakroh/kfp-test:v0.9",
    packages_to_install=[]
)
def visualize_inference(inference_results_csv:Input[Artifact],inference_results_video: Output[Artifact]):
    import pandas as pd
    import ast
    import cv2 as cv
    import numpy as np
    from PIL import Image
    import io
    import subprocess
    
    video_path = '/data/sample_videos/sample.mp4'
    output_path = inference_results_video.path + '.mp4'
    detection_results_path = inference_results_csv.path
    
    # results_df = pd.DataFrame(detection_results)
    results_df = pd.read_csv(detection_results_path)
    results_df['vehicle_bbox'] = results_df['vehicle_bbox'].apply(ast.literal_eval)
    results_df['lp_bbox'] = results_df['lp_bbox'].apply(ast.literal_eval)
    max_frame_num = results_df["frame_number"].max()
    cap = cv.VideoCapture(video_path)
    fps = cap.get(cv.CAP_PROP_FPS)
    total_frame = cap.get(cv.CAP_PROP_FRAME_COUNT)

    # output video 
    fourcc = cv.VideoWriter_fourcc(*'avc1')  # Specify the codec
    width = int(cap.get(cv.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv.CAP_PROP_FRAME_HEIGHT))
    out = cv.VideoWriter(output_path, fourcc, fps, (width, height))


    def get_bytes_from_prediction(prediction: np.ndarray,quality: int) -> bytes:
        im_rgb = prediction[...,::-1]
        return_image = Image.fromarray(im_rgb)
        return_bytes = io.BytesIO()
        return_image.save(return_bytes, format='JPEG', quality=quality)
        return_bytes.seek(0)
        return return_bytes
    
    frame_num = -1
    # max_frame_num = 10
    while cap.isOpened():
        frame_num += 1
        ret, frame = cap.read()
        ### limit 10 frame for test purpose
        if not ret or frame_num > max_frame_num:
            break
        df_ = results_df[results_df['frame_number'] == frame_num]
        for index in range(len(df_)):
            # draw vehicle bbox
            print(f"lp_number : {df_.iloc[index]['lp_number']}")
            # print(f"vehicle_bbox : {df_.iloc[index]['vehicle_bbox']}")
        
            vhcl_x1, vhcl_y1, vhcl_x2, vhcl_y2 = df_.iloc[index]['vehicle_bbox']
            cv.rectangle(frame, (int(vhcl_x1), int(vhcl_y1)),(int(vhcl_x2), int(vhcl_y2)), (0, 255, 0), 8)
            # draw license plate bbox
            plate_x1, plate_y1, plate_x2, plate_y2 = df_.iloc[index]['lp_bbox']

            # region of interest
            roi = frame[int(vhcl_y1):int(vhcl_y2), int(vhcl_x1):int(vhcl_x2)]
            cv.rectangle(roi, (int(plate_x1), int(plate_y1)), (int(plate_x2), int(plate_y2)), (0, 0, 255), 6)

            # write detected number
            (text_width, text_height), _ = cv.getTextSize(str(df_.iloc[index]['lp_number']),cv.FONT_HERSHEY_DUPLEX,2,6)
            cv.putText(
                    roi,
                    str(df_.iloc[index]['lp_number']),
                    (int((plate_x2 + plate_x1 - text_width)/2), int(plate_y1 - text_height)),
                    cv.FONT_HERSHEY_DUPLEX,
                    2,
                    (0, 255, 0),
                    6
                )
            return_bytes = get_bytes_from_prediction(frame,quality=65)
            image_np = np.frombuffer(return_bytes.read(), np.uint8)
            input_image = cv.imdecode(image_np, cv.IMREAD_COLOR) 
            
        out.write(input_image)
    out.release()
    cap.release()
    subprocess.run(['cp',output_path,inference_results_video.path])

In [129]:
@dsl.pipeline()
def lp_number_dection_video(model_url:str,vehicle_conf: float = 0.5,license_conf: float = 0.5):
    task1 = inference_video(model_url=model_url,vehicle_conf=vehicle_conf,license_conf=license_conf)
    task1.set_cpu_limit("4")
    task1.set_memory_limit("16Gi")
    kfp.kubernetes.add_pod_annotation(
        task=task1,
        annotation_key="hpe-ezua/add-auth-token",
        annotation_value="true"
    )
    kubernetes.mount_pvc(
        task1,
        pvc_name='user-pvc',
        mount_path='/data',
    )
    task2 = visualize_inference(inference_results_csv=task1.output)
    task2.set_cpu_limit("4")
    task2.set_memory_limit("16Gi")
    kfp.kubernetes.add_pod_annotation(
        task=task2,
        annotation_key="hpe-ezua/add-auth-token",
        annotation_value="true"
    )
    kubernetes.mount_pvc(
        task2,
        pvc_name='user-pvc',
        mount_path='/data',
    )

InconsistentTypeException: Incompatible argument passed to the input 'vehicle_conf' of component 'inference-video': Argument type 'NUMBER_DOUBLE' is incompatible with the input type 'NUMBER_INTEGER'

In [125]:
model_url = 'https://my-isvc-predictor-geun-tak-roh-2e590eb8.ingress.pcai0308.sg2.hpecolo.net'

kfp_client.create_run_from_pipeline_func(
    lp_number_dection_video,
    arguments={
        'model_url': model_url,
        'vehicle_conf': 0.5,
        'license_conf': 0.5,
    },
    experiment_name="test-rhgt-exp",
)

RunPipelineResult(run_id=976f1e91-8da5-4c74-900a-417e73d28fc0)

In [127]:
from urllib.parse import urlparse

model_url = 'https://my-isvc-predictor-geun-tak-roh-2e590eb8.ingress.pcai0308.sg2.hpecolo.net'

parts = urlparse(model_url)
split = parts.netloc.split('.')[0]
svc_name, namespace = split.split('-predictor-')
url_for_yolo = f"http://{svc_name}-predictor.{namespace}.svc.cluster.local"

In [128]:
from kfp import compiler, dsl

compiler.Compiler().compile(lp_number_dection_video, package_path='lp_number_dection_video.yaml')